In [0]:
# Import necessary PySpark functions for data cleansing
from pyspark.sql.functions import col, trim, upper, current_timestamp, when, lit

# Test spark session and check basic info
print("Spark Session is active and ready for Silver Layer Transformation!")

Spark Session is active and ready for Silver Layer Transformation!


In [0]:
# 1. Create Unity Catalog and Silver Schema
spark.sql("CREATE CATALOG IF NOT EXISTS atliq;")
spark.sql("CREATE SCHEMA IF NOT EXISTS atliq.silver;")

print("Unity Catalog 'atliq.silver' schema is ready!")

Unity Catalog 'atliq.silver' schema is ready!


In [0]:
bronze_customers_path = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze/customers/"

df_bronze_customers = spark.read.parquet(bronze_customers_path)

print("Raw Bronze Customers Data Preview:")
display(df_bronze_customers.limit(5))

Raw Bronze Customers Data Preview:


customer_id,customer_name,email,city,signup_date,updated_at,ingest_date
1,Nisha Gupta,nisha.gupta541@example.com,delhi,2024-01-02,2024-01-02T00:48:55.000Z,2026-10-22
2,Tara Chopra,tara.chopra529@example.com,Bengaluru,2024-01-02,2024-01-02T10:44:55.000Z,2026-10-22
3,Vivaan Chopra,vivaan.chopra414@example.com,Mumbai,2024-01-02,2024-01-02T21:00:42.000Z,2026-10-22
4,Sai Rao,sai.rao859@example.com,Mumbai,2024-01-03,2024-01-03T06:44:44.000Z,2026-10-22
5,Aman Sandhu,aman.sandhu622@example.com,Chennai,2024-01-03,2024-01-03T22:53:00.000Z,2026-10-22


In [0]:
from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze"

CITY_MAP = {"Bangalore": "Bengaluru", "Bombay": "Mumbai",
            "Calcutta": "Kolkata", "Madras": "Chennai"}

mapping = F.create_map([F.lit(x) for kv in CITY_MAP.items() for x in kv])
city_clean = F.initcap(F.trim(F.regexp_replace("city", r"\s+", " ")))

customers = (
    spark.read.parquet(f"{BRONZE}/customers")
    .withColumn("city", F.coalesce(mapping[city_clean], city_clean))
    .withColumn("signup_date", F.to_date("signup_date"))
    # is_test flag add kar rahe hain (agar email me @atliq-test.com hai toh true, warna false)
    .withColumn("is_test", F.lower(F.col("email")).rlike(r"@atliq-test\.com"))
    .dropDuplicates(["customer_id"])
    .filter(F.col("customer_id").isNotNull())
)

customers.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("atliq.silver.customers")
print("Silver 'customers' updated successfully with is_test column!")

Silver 'customers' updated successfully with is_test column!


In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql.functions import col, trim, initcap

bronze_products_path = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze/products/"

cat_clean = F.initcap(F.trim(F.regexp_replace("category", r"\s+", " ")))
cat_clean = F.regexp_replace(cat_clean, r"(?i)\s+and\s+", " & ")

df_bronze_products = spark.read.parquet(bronze_products_path)

# Keep only the latest version of each product across all Bronze batches
w = Window.partitionBy("product_id").orderBy(F.col("updated_at").desc(), F.col("ingest_date").desc())
df_latest = (df_bronze_products
             .withColumn("rn", F.row_number().over(w))
             .filter("rn = 1")
             .drop("rn"))

df_silver_products = df_latest.select(
    col("product_id"),
    initcap(trim(F.regexp_replace(col("product_name"), r"\s+", " "))).alias("product_name"),
    cat_clean.alias("category"),
    col("unit_price"),
    col("updated_at"),
    col("ingest_date")
)

df_silver_products.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("atliq.silver.products")

print("Silver 'products' cleaned, de-duplicated and saved!")

Silver 'products' cleaned, de-duplicated and saved!


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, trim, initcap

# 1. Read Bronze orders data using recursiveFileLookup to scan all partition subfolders
bronze_orders_path = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze/orders/"

df_bronze_orders = (
    spark.read.option("recursiveFileLookup", "true")
    .parquet(bronze_orders_path)
)

# 2. Data Cleansing & Deduplication
df_silver_orders_updates = df_bronze_orders.select(
    col("order_id"),
    col("customer_id"),
    col("order_date"),
    initcap(trim(col("status"))).alias("status"),
    col("order_amount"),
    col("created_at"),
    col("updated_at")
).dropDuplicates(["order_id"])

# FIX: convert to a date, then replace the legacy 1900-01-01 with the date of created_at
df_silver_orders_updates = (
    df_silver_orders_updates
    .withColumn("order_date", F.to_date("order_date"))
    .withColumn(
        "order_date",
        F.when(F.col("order_date") == F.lit("1900-01-01").cast("date"),
               F.to_date("created_at"))
         .otherwise(F.col("order_date"))
    )
)

# check before writing: this should print 0
print("Still 1900:", df_silver_orders_updates
      .filter(F.col("order_date") < F.lit("2000-01-01").cast("date")).count())


target_table_name = "atliq.silver.orders"

# 3. Delta MERGE (Upsert) into Silver Layer
if spark.catalog.tableExists(target_table_name):
    deltaTable = DeltaTable.forName(spark, target_table_name)
    
    deltaTable.alias("target").merge(
        df_silver_orders_updates.alias("source"),
        "target.order_id = source.order_id"
    ).whenMatchedUpdate(set = {
        "order_date": "source.order_date",   
        "status": "source.status",
        "order_amount": "source.order_amount",
        "updated_at": "source.updated_at"
    }).whenNotMatchedInsert(values = {
        "order_id": "source.order_id",
        "customer_id": "source.customer_id",
        "order_date": "source.order_date",
        "status": "source.status",
        "order_amount": "source.order_amount",
        "created_at": "source.created_at",
        "updated_at": "source.updated_at"
    }).execute()
    
    print("Silver 'orders' table successfully updated via Delta MERGE!")
else:
    df_silver_orders_updates.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table_name)
        
    print("Silver 'orders' table created and saved successfully!")

Still 1900: 0
Silver 'orders' table created and saved successfully!


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col, trim, initcap

bronze_payments_path = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze/payments/"

df_bronze_payments = (
    spark.read.option("recursiveFileLookup", "true")
    .parquet(bronze_payments_path)
)

# 1. Data Cleansing & Deduplication (Schema ke mutabiq 'method' column use karenge)
df_silver_payments_updates = df_bronze_payments.select(
    col("payment_id"),
    col("order_id"),
    col("amount"),
    initcap(trim(col("method"))).alias("method"),
    col("paid_at"),
    col("updated_at")
).dropDuplicates(["payment_id"])

target_payments_table = "atliq.silver.payments"

# 2. Delta MERGE (Upsert) into Silver Layer
if spark.catalog.tableExists(target_payments_table):
    deltaTable = DeltaTable.forName(spark, target_payments_table)
    
    deltaTable.alias("target").merge(
        df_silver_payments_updates.alias("source"),
        "target.payment_id = source.payment_id"
    ).whenMatchedUpdate(set = {
        "amount": "source.amount",
        "method": "source.method",
        "paid_at": "source.paid_at",
        "updated_at": "source.updated_at"
    }).whenNotMatchedInsert(values = {
        "payment_id": "source.payment_id",
        "order_id": "source.order_id",
        "amount": "source.amount",
        "method": "source.method",
        "paid_at": "source.paid_at",
        "updated_at": "source.updated_at"
    }).execute()
    
    print("Silver 'payments' table successfully updated via Delta MERGE!")
else:
    df_silver_payments_updates.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_payments_table)
        
    print("Silver 'payments' table created and saved successfully!")

Silver 'payments' table successfully updated via Delta MERGE!


In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import col

# 1. Read Bronze order_items data using recursiveFileLookup
bronze_order_items_path = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze/order_items/"

df_bronze_order_items = (
    spark.read.option("recursiveFileLookup", "true")
    .parquet(bronze_order_items_path)
)

# 2. Data Cleansing & Deduplication
df_silver_order_items_updates = df_bronze_order_items.select(
    col("order_item_id"),
    col("order_id"),
    col("product_id"),
    col("quantity"),
    col("item_price"),
    col("created_at")
).dropDuplicates(["order_item_id"])

target_order_items_table = "atliq.silver.order_items"

# 3. Delta MERGE (Upsert) into Silver Layer
if spark.catalog.tableExists(target_order_items_table):
    deltaTable = DeltaTable.forName(spark, target_order_items_table)
    
    deltaTable.alias("target").merge(
        df_silver_order_items_updates.alias("source"),
        "target.order_item_id = source.order_item_id"
    ).whenMatchedUpdate(set = {
        "order_id": "source.order_id",
        "product_id": "source.product_id",
        "quantity": "source.quantity",
        "item_price": "source.item_price",
        "created_at": "source.created_at"
    }).whenNotMatchedInsert(values = {
        "order_item_id": "source.order_item_id",
        "order_id": "source.order_id",
        "product_id": "source.product_id",
        "quantity": "source.quantity",
        "item_price": "source.item_price",
        "created_at": "source.created_at"
    }).execute()
    
    print("Silver 'order_items' table successfully updated via Delta MERGE!")
else:
    df_silver_order_items_updates.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_order_items_table)
        
    print("Silver 'order_items' table created and saved successfully!")

Silver 'order_items' table successfully updated via Delta MERGE!


In [0]:
from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze"

SCHEMA = ("event_id STRING, session_id STRING, customer_id BIGINT, event_type STRING, "
          "event_ts STRING, product_id BIGINT, order_id BIGINT, device STRING, "
          "user_agent STRING, utm_source STRING")

# 1. Read raw clickstream lines from Bronze
lines = spark.read.text(f"{BRONZE}/clickstream")
parsed = lines.withColumn("e", F.from_json("value", SCHEMA))

# 2. Send unparseable lines to quarantine table
(parsed.filter("e.event_id IS NULL").select("value")
   .write.mode("overwrite").saveAsTable("atliq.silver.web_events_quarantine"))

# 3. Helper normalization function
def norm(col):
    return F.lower(F.regexp_replace(F.trim(col), r"\s+", " "))

# 4. Parse timestamps (epoch milliseconds or ISO text)
ts = (F.when(F.col("event_ts").rlike(r"^\d+$"),
              F.timestamp_millis(F.col("event_ts").cast("bigint")))
      .otherwise(F.try_to_timestamp("event_ts", F.lit("yyyy-MM-dd'T'HH:mm:ss.SSSX"))))

ev = (parsed.filter("e.event_id IS NOT NULL").select("e.*")
        .withColumn("event_ts", ts)
        .withColumn("event_type", norm(F.col("event_type")))
        .dropDuplicates(["event_id"])
        .filter(F.col("session_id").isNotNull() & (F.trim("session_id") != "")))

# 5. Filter out bot sessions and unknown products
bots = (ev.filter(F.col("user_agent").rlike(r"(?i)(bot|crawler|spider|python-requests)"))
          .select("session_id").distinct())

known = spark.table("atliq.silver.products").select(F.col("product_id").alias("_pid"))

ev = (ev.join(bots, "session_id", "left_anti")
        .join(known, F.col("product_id") == F.col("_pid"), "left")
        .filter(F.col("product_id").isNull() | F.col("_pid").isNotNull()).drop("_pid"))

# 6. Save clean events to Silver delta table
ev.write.mode("overwrite").saveAsTable("atliq.silver.web_events")

print("Clickstream processing & quarantine successfully completed!")

Clickstream processing & quarantine successfully completed!


In [0]:
from pyspark.sql import functions as F

BRONZE = "abfss://lakehouse@stgatiqliqlakehouse.dfs.core.windows.net/bronze"

# ==========================================
# 1. Supplier Price List -> Silver
# ==========================================
sup_path = f"{BRONZE}/supplier_price_list"
sup = spark.read.option("header", "true").parquet(sup_path)

# Defensively extract amount avoiding regex trap
amount_expr = F.regexp_replace(
    F.regexp_extract("supplier_cost", r"-?[0-9][0-9,]*(\.[0-9]+)?", 0), ",", ""
)

sup_cleaned = (
    sup.withColumn("supplier_cost", F.when(amount_expr != "", amount_expr.cast("decimal(12,2)")))
       .withColumn("effective_date", F.coalesce(
           *[F.to_date(F.try_to_timestamp("effective_date", F.lit(f)))
             for f in ["yyyy-MM-dd", "dd/MM/yyyy", "dd-MMM-yyyy"]]
       ))
)

before_sup = sup_cleaned.count()
sup_final = sup_cleaned.dropDuplicates()
sup_final.write.format("delta").mode("overwrite").saveAsTable("atliq.silver.supplier_price_list")
print(f"Silver 'supplier_price_list' created successfully! Exact duplicates removed: {before_sup - sup_final.count()}")


# ==========================================
# 2. Marketing Spend -> Silver
# ==========================================
mkt_path = f"{BRONZE}/marketing_spend"
mkt = spark.read.option("header", "true").parquet(mkt_path)

CHANNEL_MAP = {
    "meta ads": "Meta Ads", "facebook": "Meta Ads", "fb": "Meta Ads",
    "google ads": "Google Ads", "google": "Google Ads",
    "email": "Email", "seo": "SEO", "influencer": "Influencer"
}

def norm_channel(col_name):
    lower_c = F.lower(F.trim(col_name))
    expr = F.lit(None).cast("string")
    for k, v in CHANNEL_MAP.items():
        expr = F.when(lower_c.contains(k), F.lit(v)).otherwise(expr)
    return F.coalesce(expr, F.lit("Other"))

spend_amount = F.regexp_replace(F.regexp_replace("spend_amount", "[^0-9.-]", ""), ",", "")
clicks_count = F.regexp_replace(F.regexp_replace("clicks", "[^0-9]", ""), ",", "")

mkt_cleaned = (
    mkt.withColumn("channel", norm_channel(F.col("channel")))
       .withColumn("spend_amount", F.when(spend_amount != "", spend_amount.cast("decimal(12,2)")))
       .withColumn("clicks", F.when(clicks_count != "", clicks_count.cast("long")))
       .withColumn("spend_date", F.to_date("spend_date"))
       .filter(F.col("spend_amount").isNotNull())
)

before_mkt = mkt_cleaned.count()
mkt_final = mkt_cleaned.dropDuplicates()
mkt_final.write.format("delta").mode("overwrite").saveAsTable("atliq.silver.marketing_spend")
print(f"Silver 'marketing_spend' created successfully! Rows processed: {mkt_final.count()}")

Silver 'supplier_price_list' created successfully! Exact duplicates removed: 4
Silver 'marketing_spend' created successfully! Rows processed: 3037
